In [ ]:
import json, requests, os
import urllib.request
import shutil
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import re

In [34]:
#import set

set_code = "EOE"
url = f"https://api.scryfall.com/cards/search?q=set:{set_code}&unique=prints"

cards = []
has_more = True
while has_more:
    response = requests.get(url)
    data = response.json()
    cards.extend(data["data"])
    has_more = data.get("has_more", False)
    if has_more:
        url = data["next_page"]

#save metadata
os.makedirs(f"data/{set_code}/meta", exist_ok=True)
with open(f"data/{set_code}/meta/cards.json", "w") as f:
    json.dump(cards, f, indent = 2)


print(f"downloaded {len(cards)} cards.")

downloaded 399 cards.


In [35]:
#test loading of JSON

with open(f"data/{set_code}/meta/cards.json", "r") as f:
    json.load(f)

print(type(cards))
print(len(cards))
print(cards[0].keys())

def find_by_name(cards, name):
    results = [c for c in cards if c.get("name", "").lower() == name.lower()]
    return results

def safe_filename(name: str) -> str:
    # Replace any character that's not alphanumeric, space, dot, or underscore
    return re.sub(r'[^a-zA-Z0-9 _.-]', '_', name)

#find_by_name(cards, "ragost, deft gastronaut")

<class 'list'>
399
dict_keys(['object', 'id', 'oracle_id', 'multiverse_ids', 'mtgo_id', 'tcgplayer_id', 'cardmarket_id', 'name', 'lang', 'released_at', 'uri', 'scryfall_uri', 'layout', 'highres_image', 'image_status', 'image_uris', 'mana_cost', 'cmc', 'type_line', 'oracle_text', 'colors', 'color_identity', 'keywords', 'produced_mana', 'all_parts', 'legalities', 'games', 'reserved', 'game_changer', 'foil', 'nonfoil', 'finishes', 'oversized', 'promo', 'reprint', 'variation', 'set_id', 'set', 'set_name', 'set_type', 'set_uri', 'set_search_uri', 'scryfall_set_uri', 'rulings_uri', 'prints_search_uri', 'collector_number', 'digital', 'rarity', 'card_back_id', 'artist', 'artist_ids', 'illustration_id', 'border_color', 'frame', 'security_stamp', 'full_art', 'textless', 'booster', 'story_spotlight', 'edhrec_rank', 'preview', 'prices', 'related_uris', 'purchase_uris'])


In [36]:
#download images of cards in JSON

image_dir = f"data/{set_code}/images"
os.makedirs(image_dir, exist_ok=True)

for card in cards:
    if "image_uris" in card: 
        name = safe_filename(card["name"])
        set_name = card["set"].upper()
        collector = card.get("collector_number", "NA")
        filename = f"{name}_{set_name}_{collector}"
        img_url = card["image_uris"]["normal"]
        out_path = os.path.join(image_dir, f"{filename}.jpg")
        if not os.path.exists(out_path):
            urllib.request.urlretrieve(img_url, out_path)

In [37]:
#turn into class datasets

dataset_dir = f"data/{set_code}/dataset"
os.makedirs(dataset_dir, exist_ok=True)

for card in cards:
    set_name = card["set"].upper()
    collector = card.get("collector_number", "NA")

    if "image_uris" in card:
        name = safe_filename(card["name"])
        collector = card.get("collector_number", "NA")
        filename = f"{name}_{set_name}_{collector}"
        src = os.path.join(image_dir, f"{filename}.jpg")
        dst_dir = os.path.join(dataset_dir, filename)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, f"{filename}.jpg")
        if not os.path.exists(dst):
            shutil.copy(src, dst)

In [ ]:
#CNN

num_classes = len(cards)

model = models.resnet18(pretrained = True)
model.fc = nn.Linear(model.fc.in_features, num_classes)

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    #Geometric
    transforms.RandomRotation(degrees = 5),
    transforms.RandomPerspective(distortion_scale=0.1, p = 0.5),
    transforms.RandomAffine(degrees = 0, translate = (0.05, 0.05)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),

    #Photometric
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.1, hue = 0.02),

    #Blur & Noise
    transforms.RandomApply([transforms.GaussianBlur(kernel_size = 3)], p = 0.3),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),

    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(f"data/{set_code}/dataset")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_subset, val_subset = random_split(dataset, [train_size, val_size])

train_set = datasets.ImageFolder(f"data/{set_code}/dataset", transform=train_transform)
val_set   = datasets.ImageFolder(f"data/{set_code}/dataset", transform=val_transform)

train_set.samples = [dataset.samples[i] for i in train_subset.indices]
val_set.samples   = [dataset.samples[i] for i in val_subset.indices]


train_loader = DataLoader(train_set, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_set, batch_size = 32)

c:\Users\marti\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\marti\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


TypeError: Unexpected type <class 'torch.utils.data.dataset.Subset'>

In [71]:
#GPT attempt

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
import random

# ------------------------
# Hyperparameters
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(cards)  # your cards list
batch_size = 32
epochs = 5
repeats_per_card = 3
lr = 1e-4

# ------------------------
# Transforms
# ------------------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    # Geometric
    transforms.RandomRotation(5),
    transforms.RandomPerspective(distortion_scale=0.1, p=0.5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),

    # Photometric
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),

    # Blur
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),

    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ------------------------
# Augmented dataset
# ------------------------
class AugmentedDataset(Dataset):
    def __init__(self, base_dataset, transform, repeats=3):
        self.base = base_dataset
        self.transform = transform
        self.repeats = repeats

    def __len__(self):
        return len(self.base) * self.repeats

    def __getitem__(self, idx):
        orig_idx = idx % len(self.base)
        img, label = self.base[orig_idx]  # PIL Image
        img = self.transform(img)
        return img, label

# ------------------------
# Load base dataset
# ------------------------
base_dataset = datasets.ImageFolder("data/EOE/dataset", transform=None)

# Training dataset with repeats and augmentation
train_dataset = AugmentedDataset(base_dataset, train_transform, repeats=repeats_per_card)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Evaluation dataset: small random subset
val_subset_indices = random.sample(range(len(base_dataset)), min(32, len(base_dataset)))
val_dataset = Subset(base_dataset, val_subset_indices)

# Wrapper to apply val_transform
class ValWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        img = self.transform(img)
        return img, label

val_dataset = ValWrapper(val_dataset, val_transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ------------------------
# Model setup
# ------------------------
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Scheduler: reduce LR on plateau
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# ------------------------
# Training loop
# ------------------------
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    # Scheduler step
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"train_loss={avg_train_loss:.4f} | "
          f"val_loss={avg_val_loss:.4f} | "
          f"val_acc={val_acc:.3f} | "
          f"lr={optimizer.param_groups[0]['lr']:.1e}")


Epoch 1/5 | train_loss=5.4721 | val_loss=4.6811 | val_acc=0.312 | lr=1.0e-04
Epoch 2/5 | train_loss=4.1787 | val_loss=3.2686 | val_acc=0.844 | lr=1.0e-04
Epoch 3/5 | train_loss=3.0469 | val_loss=2.1567 | val_acc=1.000 | lr=1.0e-04
Epoch 4/5 | train_loss=2.1250 | val_loss=1.2011 | val_acc=1.000 | lr=1.0e-04
Epoch 5/5 | train_loss=1.4006 | val_loss=0.7771 | val_acc=1.000 | lr=1.0e-04


In [72]:
torch.save(model.state_dict(), "card_model_eoe.pth")

idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
with open("idx_to_class.json", "w") as f:
    json.dump(idx_to_class, f, indent = 2)

In [ ]:
#testing

import torch
from torchvision import transforms, models
from PIL import Image
import json

# --- 1. Load the model ---
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
num_classes = len(idx_to_class)  # make sure this matches your training
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("card_model_eoe.pth", map_location="cpu"))  # or device if using GPU
model.eval()

# --- 2. Load the index->class mapping ---
with open("idx_to_class.json", "r") as f:
    idx_to_class = json.load(f)

# --- 3. Define the same transforms used in training ---
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- 4. Load your test image ---
image_path = "test_EOE1.jpg"  # replace with your filename
img = Image.open(image_path).convert("RGB")
img_tensor = val_transform(img).unsqueeze(0)  # add batch dimension

# --- 5. Make prediction ---
with torch.no_grad():
    outputs = model(img_tensor)
    probs = torch.nn.functional.softmax(outputs, dim=1)
    pred_idx = torch.argmax(probs, dim=1).item()
    pred_class = idx_to_class[str(pred_idx)]
    confidence = probs[0, pred_idx].item()

print(f"Predicted card: {pred_class} (confidence: {confidence:.2f})")


In [38]:
#complete? run file for classification

with open("idx_to_class.json", "r") as f:
    idx_to_class = json.load(f)

num_classes = len(idx_to_class)
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("card_model_eoe.pth", map_location="cpu"))
model.eval()

# ------------------------
# Transform (same as training)
# ------------------------
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ------------------------
# Utility functions
# ------------------------
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # top-left
    rect[2] = pts[np.argmax(s)]  # bottom-right
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # top-right
    rect[3] = pts[np.argmax(diff)]  # bottom-left
    return rect

def predict_card(cv2_img):
    img = cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img)
    img_tensor = val_transform(pil_img).unsqueeze(0)
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()
        pred_class = idx_to_class[str(pred_idx)]
        confidence = probs[0, pred_idx].item()
    return pred_class, confidence

def draw_and_warp(img, threshold_1=50, threshold_2=150, tolerance=0.2, min_area_frac=0.2):
    img_height, img_width = img.shape[:2]
    magic_aspect = 6.35/8.89
    min_area = min_area_frac * img_height * img_width

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, threshold_1, threshold_2)
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    warped = None
    for cnt in contours:
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2)
            rect = order_points(pts)
            (tl, tr, br, bl) = rect
            w = np.linalg.norm(br - bl)
            h = np.linalg.norm(tr - br)
            aspect_ratio = w / h if h != 0 else 0
            area = cv2.contourArea(rect)
            if magic_aspect - tolerance <= aspect_ratio <= magic_aspect + tolerance and area > min_area:
                cv2.drawContours(img, [approx], -1, (255, 255, 0), 2)
                dst_height, dst_width = 680, 488
                dst = np.array([[0, 0], [dst_width - 1, 0],
                                [dst_width - 1, dst_height - 1], [0, dst_height - 1]], dtype="float32")
                M = cv2.getPerspectiveTransform(rect, dst)
                warped = cv2.warpPerspective(img, M, (dst_width, dst_height))
                break
    return img, warped

# ------------------------
# Main webcam loop (with SSIM gating)
# ------------------------
capture = cv2.VideoCapture(0)

last_warped_gray = None
pred_class_old = None
SIMILARITY_THRESHOLD = 0.9  # higher = stricter
conf_threshold = 0.05

while True:
    ret, frame = capture.read()
    if not ret:
        break

    contoured, warped = draw_and_warp(frame)
    cv2.imshow("card scanner", contoured)

    if warped is not None:
        cv2.imshow("warped", warped)
        gray_current = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

        if last_warped_gray is None:
            # First detection → always predict
            pred_class, confidence = predict_card(warped)
            if pred_class != pred_class_old and confidence > conf_threshold:
                print(f"Predicted: {pred_class} | conf: {confidence:.2f}")
                pred_class_old = pred_class
                last_warped_gray = gray_current

        else:
            # Compare to last frame using SSIM
            score = ssim(gray_current, last_warped_gray)
            if score < SIMILARITY_THRESHOLD:  # card changed significantly
                pred_class, confidence = predict_card(warped)
                if pred_class != pred_class_old and confidence > conf_threshold:
                    print(f"Predicted: {pred_class} | conf: {confidence:.4f}")
                    pred_class_old = pred_class
                    last_warped_gray = gray_current



        
        

    else:
        last_warped_gray = None  # reset when no card detected

    if cv2.waitKey(1) & 0xff == ord('q'):
        break

capture.release()
cv2.destroyAllWindows()


c:\Users\marti\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\marti\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Predicted: Systems Override | conf: 0.31
Predicted: Wurmwall Sweeper | conf: 0.37


things missing:

1. look into incremental training; if a new set releases, how can the model only be tweaked to add new cards instead of requiring full retrain?

2. go over everything, make sure you truly understand everything that goes on

3. general cleanup; remove unwanted files, consolidate imports, split main into multiple files? learn how to import sections from other files, not just from X import *

4. look into the confidence as it is still very low, very weird

5. expand card database, retrain model for more cards

6. look into identical cards and setting default version, ex. regular vs special foil